# Problem 28 — 3Sum

- **LeetCode**: [#15](https://leetcode.com/problems/3sum/)
- **Difficulty**: Medium
- **Pattern**: **Sort + Outer fix-one loop + Two Pointers converging**
- **Target time**: 35-45 min

## Problem
Given an integer array `nums`, return **all unique triplets** `[nums[i], nums[j], nums[k]]` such that `i != j != k` and `nums[i] + nums[j] + nums[k] == 0`. The result set must not contain duplicate triplets.

**Examples**
- `[-1, 0, 1, 2, -1, -4]` → `[[-1, -1, 2], [-1, 0, 1]]`
- `[0, 1, 1]` → `[]`
- `[0, 0, 0]` → `[[0, 0, 0]]`

**Constraints**: `3 <= n <= 3000`, `-10^5 <= nums[i] <= 10^5`.

## The reduction (why this is 2Sum in disguise)

Fix one element `nums[i]`. The remaining task is: in `nums[i+1:]`, find two numbers that sum to `-nums[i]`. That's **2Sum on a sorted slice** — Problem 13's pattern: converging pointers, decide by sum vs target.

```
sort nums
for i in 0..n-3:
    if nums[i] == nums[i-1]: skip                # dedupe outer
    if nums[i] > 0: break                        # smallest sum already > 0
    l, r = i+1, n-1
    target = -nums[i]
    while l < r:                                 # two-pointer 2Sum on sorted
        s = nums[l] + nums[r]
        if s == target: record, skip l/r dupes, advance both
        elif s < target: l += 1
        else: r -= 1
```

**Why sort?** Two reasons stacked:
1. Sorted order lets two pointers run in O(n) per outer iteration → total O(n²).
2. Sorted order makes deduplication trivial: equal values sit next to each other, so `nums[i] == nums[i-1]` skip handles uniqueness without a `set`.

## Dedup carefully — the three layers

1. **Outer `i`**: skip if `nums[i] == nums[i-1]` (and `i > 0`). Without this, `[-1,-1,0,1]` would emit `[-1,0,1]` twice.
2. **Inner `left`** after a match: advance past equal values. `[-2,0,0,0,2]` would otherwise emit `[-2,0,2]` three times.
3. **Inner `right`** after a match: same, mirror side.

Skipping dupes only **after a match** (not before) is the cleanest invariant — it keeps the pointer movement monotone.

In [27]:
class Solution:
    def threeSum(self, nums):
        res = []
        n = len(nums)

        nums.sort()

        for i in range(0,n - 2):
            if i > 0 and nums[i] == nums[i - 1]:
                continue

            low,high = i+1,n-1
            while(low<high):
                total_sum = nums[i] + nums[low] + nums[high]

                if total_sum == 0:
                    res.append([nums[i],nums[low],nums[high]])
                    low += 1
                    high -= 1
                    while low < high and nums[low] == nums[low - 1]:
                        low += 1

                    while low < high and nums[high] == nums[high + 1]:
                        high -= 1
                elif total_sum > 0:
                    high -= 1
                else:
                    low += 1
        return res

In [28]:
ob = Solution()
nums = [-4, -2, -2, -2, 0, 1, 2, 2, 2, 3, 3, 4, 4, 6, 6]
print(ob.threeSum(nums))

[[-4, -2, 6], [-4, 0, 4], [-4, 1, 3], [-4, 2, 2], [-2, -2, 4], [-2, 0, 2]]


In [29]:
def normalize(triplets):
    # order-independent comparison: sort each triplet, then sort the list
    return sorted(sorted(t) for t in triplets)

def test(nums, expected):
    got = Solution().threeSum(list(nums))
    assert normalize(got) == normalize(expected), f'fail on {nums}: got {got}, expected {expected}'
    print(f'OK  {str(nums):>28}  ->  {got}')

test([-1, 0, 1, 2, -1, -4], [[-1, -1, 2], [-1, 0, 1]])
test([0, 1, 1],              [])                          # no valid triplet
test([0, 0, 0],              [[0, 0, 0]])                 # single triplet of zeros
test([0, 0, 0, 0],           [[0, 0, 0]])                 # duplicate handling — only one triplet
test([-2, 0, 1, 1, 2],       [[-2, 0, 2], [-2, 1, 1]])    # left-side duplicates as part of valid triplet
test([1, 2, 3],              [])                          # all positive — early break
test([-1, -1, 2, 2],         [[-1, -1, 2]])               # dupe on both ends, one valid triplet
test([-4, -2, -2, -2, 0, 1, 2, 2, 2, 3, 3, 4, 4, 6, 6],
     [[-4, -2, 6], [-4, 0, 4], [-4, 1, 3], [-4, 2, 2], [-2, -2, 4], [-2, 0, 2]])
test([],                     [])                          # empty
test([0],                    [])                          # too small
test([-1, 0, 1],             [[-1, 0, 1]])                # minimum-size valid input


OK         [-1, 0, 1, 2, -1, -4]  ->  [[-1, -1, 2], [-1, 0, 1]]
OK                     [0, 1, 1]  ->  []
OK                     [0, 0, 0]  ->  [[0, 0, 0]]
OK                  [0, 0, 0, 0]  ->  [[0, 0, 0]]
OK              [-2, 0, 1, 1, 2]  ->  [[-2, 0, 2], [-2, 1, 1]]
OK                     [1, 2, 3]  ->  []
OK                [-1, -1, 2, 2]  ->  [[-1, -1, 2]]
OK  [-4, -2, -2, -2, 0, 1, 2, 2, 2, 3, 3, 4, 4, 6, 6]  ->  [[-4, -2, 6], [-4, 0, 4], [-4, 1, 3], [-4, 2, 2], [-2, -2, 4], [-2, 0, 2]]
OK                            []  ->  []
OK                           [0]  ->  []
OK                    [-1, 0, 1]  ->  [[-1, 0, 1]]


## Pattern & Complexity

| | |
|---|---|
| **Pattern** | Sort, then for each `i` run 2Sum (converging two pointers) on the suffix |
| **Time**    | O(n²) — sort is O(n log n), then n outer × O(n) inner |
| **Space**   | O(1) extra (excluding output and sort's internal stack) |

## Related
- **#1 Two Sum** — unsorted, hashmap version. The pattern split: sorted → two pointers, unsorted → hashmap.
- **#167 Two Sum II** — sorted input, converging two pointers. The inner-loop primitive used here.
- **#16 3Sum Closest** — same outer loop, but track closest sum to target instead of equality.
- **#18 4Sum** — one more nested outer loop → O(n³). Same dedup discipline.
- **#11 Container With Most Water** — same converging-pointer template, geometric decision rule.